In [1]:
!pip install transformers

In [2]:
import torch
from transformers import GPT2Tokenizer
import pandas as pd

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
model_path = "/content/drive/MyDrive/SRDC/srdc_gpt2_final"
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

In [5]:
data_path = "/content/drive/MyDrive/SRDC/Datasets/after_feature_internal_semantic_process_data.csv"
df = pd.read_csv(data_path)
print(df.shape)
df.head()

(1524, 8)


,family,apiFeatures,dropFeatures,regFeatures,filesFeatures,filesEXTFeatures,dirFeatures,strFeatures
0,2,API CreateThread. API NtOpenKey. API GetSystem...,NaN,read registry HKEY_LOCAL_MACHINE\SAM\SAM\Domai...,NaN,NaN,NaN,STR:44;mscoree.dll. STR:1073;EncryptMode. STR:...
1,3,API GetSystemDirectory. API NtOpenFile. API Ge...,NaN,opened registry HKEY_CURRENT_USER\Software\Mic...,FILES:OPENED:C:\Documents and Settings\MyUser\...,FILES_EXT:OPENED:Manifest. FILES_EXT:OPENED:ex...,NaN,STR:6453;HELP_DECRYPT.HTML. STR:9474;HELP_DECR...
2,2,API CreateThread. API NtOpenKey. API GetSystem...,NaN,opened registry HKEY_CURRENT_USER\SOFTWARE\ODB...,NaN,NaN,NaN,STR:41;PSAPI.DLL. STR:44;mscoree.dll. STR:1466...
3,5,API GetSystemInfo. API GetSystemDirectory. API...,NaN,opened registry HKEY_CURRENT_USER\Software\Mic...,NaN,NaN,NaN,STR:44;mscoree.dll. STR:549;Warning. STR:565;W...
4,7,API GetSystemDirectory. API NtOpenFile. API Wr...,dropped file extension bat. dropped file exten...,opened registry HKEY_LOCAL_MACHINE\SYSTEM\. op...,FILES:DELETED:C:\Documents and Settings\MyUser...,FILES_EXT:DELETED:Identifier. FILES_EXT:DELETE...,DIR:CREATED:C:\Documents and Settings\MyUser\L...,NaN


In [6]:
class SRDCDataset(torch.utils.data.Dataset):

    def __init__(self, dataframe):
        dataframe = dataframe.fillna("")
        texts = dataframe[
            [
                "apiFeatures",
                "dropFeatures",
                "regFeatures",
                "filesFeatures",
                "filesEXTFeatures",
                "dirFeatures",
                "strFeatures"
            ]
        ].values
        tokenized_texts = []
        for sublist in texts:
            tokens = tokenizer(
                sublist.tolist(),
                padding="max_length",
                max_length=MAX_SEQ_LEN,
                truncation=True,
                return_tensors="pt"
            )
            tokenized_texts.append(tokens)
        self.texts = tokenized_texts
        self.labels = dataframe["family"].values
        assert len(self.texts) == len(self.labels), "Text and label count mismatch"

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        return text, label

    def __len__(self):
        return len(self.labels)

In [9]:
dataset = SRDCDataset(df)
print("Dataset size:", len(dataset))

Dataset size: 1524


In [10]:
sample_text, sample_label = dataset[0]

print(sample_text)
print("Label:", sample_label)

{'input_ids': tensor([[17614, 13610, 16818,  ..., 50256, 50256, 50256],
        [50256, 50256, 50256,  ..., 50256, 50256, 50256],
        [  961, 20478,   367,  ..., 50256, 50256, 50256],
        ...,
        [50256, 50256, 50256,  ..., 50256, 50256, 50256],
        [50256, 50256, 50256,  ..., 50256, 50256, 50256],
        [18601,    25,  2598,  ..., 50256, 50256, 50256]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}
Label: 2


In [22]:
# ===============================
# Training Configuration
# ===============================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 6
MAX_SEQ_LEN = 768
EPOCHS = 5
LR = 2e-5
K_FOLDS = 4

print("Device:", DEVICE)

Device: cuda


In [23]:
import torch
import torch.nn as nn
from transformers import GPT2Model

class AttentionClassifier(nn.Module):

    def __init__(self, hidden_size, num_classes, gpt_model_name):
        super().__init__()

        self.gpt2 = GPT2Model.from_pretrained(gpt_model_name)

        # Freeze all GPT2 layers
        for param in self.gpt2.parameters():
            param.requires_grad = False

        # Train ONLY last transformer block
        for param in self.gpt2.h[-1].parameters():
            param.requires_grad = True

        # Attention pooling
        self.attn = nn.Linear(hidden_size, 1)

        # Improved classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 7, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def attention_pool(self, x):

        weights = torch.softmax(self.attn(x), dim=1)
        pooled = torch.sum(weights * x, dim=1)

        return pooled

    def forward(self, input_ids, mask):

        groups = torch.split(input_ids, 1, dim=1)
        masks = torch.split(mask, 1, dim=1)

        outputs = []

        for g, m in zip(groups, masks):

            g = g.squeeze(1)
            m = m.squeeze(1)

            h = self.gpt2(
                input_ids=g,
                attention_mask=m
            ).last_hidden_state

            pooled = self.attention_pool(h)

            outputs.append(pooled)

        x = torch.cat(outputs, dim=1)

        return self.classifier(x)

In [24]:
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch.optim import Adam

def train_model(model, train_dataset):

    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    optimizer = Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    model.train()

    for epoch in range(EPOCHS):

        total_loss = 0

        for batch, labels in tqdm(loader):

            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(input_ids, mask)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} | Loss {total_loss/len(loader):.4f}")

In [25]:
from sklearn.metrics import classification_report, accuracy_score

def evaluate(model, test_dataset):

    loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    model.eval()

    preds = []
    truths = []

    with torch.no_grad():

        for batch, labels in loader:

            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)

            outputs = model(input_ids, mask)

            pred = torch.argmax(outputs, dim=1).cpu().numpy()

            preds.extend(pred)
            truths.extend(labels.numpy())

    acc = accuracy_score(truths, preds)

    print("\nTest Accuracy:", acc)

    print(classification_report(truths, preds))

In [26]:
from sklearn.model_selection import StratifiedKFold

def run_kfold(df):

    X = df.drop("family", axis=1)
    y = df["family"]

    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

    fold = 1

    for train_idx, test_idx in skf.split(X, y):

        print(f"\n===== Fold {fold}/{K_FOLDS} =====")

        train_df = df.iloc[train_idx]
        test_df = df.iloc[test_idx]

        train_dataset = SRDCDataset(train_df)
        test_dataset = SRDCDataset(test_df)

        model = AttentionClassifier(
            hidden_size=768,
            num_classes=12,
            gpt_model_name="/content/drive/MyDrive/SRDC/srdc_gpt2_final"
        ).to(DEVICE)

        train_model(model, train_dataset)

        evaluate(model, test_dataset)

        fold += 1

    return model

In [27]:
model = run_kfold(df)


===== Fold 1/2 =====


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

100%|██████████| 127/127 [05:42<00:00,  2.70s/it]


Epoch 1 | Loss 1.5816


100%|██████████| 127/127 [05:41<00:00,  2.69s/it]


...

100%|██████████| 127/127 [05:41<00:00,  2.69s/it]


Epoch 4 | Loss 1.0477


100%|██████████| 127/127 [05:40<00:00,  2.68s/it]


Epoch 5 | Loss 1.0036

Test Accuracy: 0.7047244094488189
              precision    recall  f1-score   support

           0       0.82      0.96      0.88       471
           1       0.87      0.52      0.65        25
           2       0.34      0.47      0.39        53
           3       0.00      0.00      0.00        23
           4       0.00      0.00      0.00        13
           5       0.27      0.69      0.39        32
           6       0.55      0.25      0.34        48
           7       0.00      0.00      0.00        30
           8       0.00      0.00      0.00         2
           9       0.82      0.31      0.45        45
          10       0.00      0.00      0.00         3
          11       0.00      0.00      0.00        17

    accuracy                           0.70       762
   macro avg       0.30      0.27      0.26       762
weighted avg       0.65      0.70      0.66       762



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [1]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/SRDC/RansomwareFamilyDetection/ransomware_family_classifier.pt"
)

NameError: name 'torch' is not defined